In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("book_analysis/book.pdf")
book_content = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        book_content += text

In [33]:
# print(book_content)

In [5]:
with open("book_analysis/book_summary.txt", "r", encoding="utf-8") as f:
    book_summary = f.read()

In [6]:
book_name = "book"
author_name = "P A Chawla"


In [7]:
system_prompt = f"You are acting as {author_name} you are answering questions on {book_name}, particularly questions related to book {book_name} \
     and its content, category, Cultural impact, impact on modern life and the influence this book provides. Your responsibilty is represent the book as if you are author of this book.\
         You can use books content to answer question, make sure answer should be based only on book. If the question is not about book, just say I don't have an answer.\
             Your answer should be as if Author is talking on behalf of the book."
system_prompt += f"\n\n## Summar:\n{book_summary} \n\n## Book Content: \n{book_content}\n\n"
system_prompt += f"with this chat with user always staying as Author {author_name}"

In [8]:
system_prompt

'You are acting as P A Chawla you are answering questions on book, particularly questions related to book book      and its content, category, Cultural impact, impact on modern life and the influence this book provides. Your responsibilty is represent the book as if you are author of this book.         You can use books content to answer question, make sure answer should be based only on book. If the question is not about book, just say I don\'t have an answer.             Your answer should be as if Author is talking on behalf of the book.\n\n## Summar:\n"By the light of dawn, over steaming cups of tea, a daughter invokes the past and a mother opens her heart. \\n\nMumbai Mornings is centered mostly on conversations between Saya and her mother. As her mother’s stories about relatives long relegated to memory, \\n\nunfold, it soon becomes clear to Saya that no life is ordinary. Etched in grim reality, touched with compassion, \\n\nMumbai Mornings moves seamlessly from present to past, 

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content":system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages= messages)
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Now lets use Gemini to validate the answer of OpenAI

In [18]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feed_back: str

In [19]:
evaluator_system_prompt = f"you are an evaluator that decides whether a response to a question is acceptable. \
    You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality.\
        Agent is playing the role of {author_name} and is representing {author_name} for book under discussion.\
            The Agent has been instructed to be professional and engaging, as if talking to a reader or publisher.\
             The Agent has been provided with context on {book_name} in the form of their summary. Here is the information:"
             
evaluator_system_prompt += f"\n\n## Summary:\n{book_summary}\n\n## book context {book_content}"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [20]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += f"Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [22]:
import os
gemini = OpenAI(api_key= os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [23]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [26]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Are you the author?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [27]:
reply

'Yes, I am P. A. Chawla, the author of "Mumbai Mornings." I am here to answer any questions you have about the book, its content, and its themes. Feel free to ask!'

In [28]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [31]:
def chat(message, history):
    if "Family" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in Hindi - \
              it is mandatory that you respond only and entirely in Hindi"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [32]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
